# 01 - Data acquisition & exploratory analysis

Goal: understand what we're trading on. We pulled US equity daily bars from Alpaca (IEX feed, free tier) for a sector-diverse 45-name universe + SPY benchmark.

This notebook checks data quality, return distributions, cross-sectional dispersion, and macro-regime context.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from mfa.config import load_config
from mfa.viz import apply_style
apply_style()

cfg = load_config(ROOT / "configs" / "default.yaml")
print("project:", cfg.project_root.name, "| processed:", cfg.processed_dir.exists())


In [ ]:
from mfa.data import build_panels

def _load_raw_panels():
    out = {}
    for k in ["open","high","low","close","volume","vwap","dollar_volume"]:
        p = cfg.processed_dir / f"panel_{k}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            df.index = pd.to_datetime(df.index, utc=True)
            out[k] = df
    if not out:
        raise RuntimeError("No cached panels found. Run scripts/01_download_data.py first.")
    return out

def _load_factors():
    d = cfg.processed_dir / "factors"
    out = {}
    for p in sorted(d.glob("*.parquet")):
        df = pd.read_parquet(p)
        df.index = pd.to_datetime(df.index, utc=True)
        out[p.stem] = df
    return out

raw_panels = _load_raw_panels()
benchmark = cfg.section("universe").get("benchmark", "SPY")
panels = build_panels(raw_panels, benchmark=benchmark, min_history_days=252)
factors = _load_factors()
print(f"loaded {len(panels)} panels and {len(factors)} factors")
print("date range:", panels["close"].index.min().date(), "->", panels["close"].index.max().date())
print("universe size:", panels["close"].shape[1], " benchmark:", benchmark)


## 1. Data quality -- missingness & survivorship

In [ ]:
close = panels["close"]
print("shape:", close.shape, "  bars:", close.shape[0], " names:", close.shape[1])
print("first 5 with most NaNs:")
print(close.isna().sum().sort_values(ascending=False).head(5))
print()
print("benchmark first/last:", panels["benchmark_close"].dropna().index[[0,-1]].date.tolist())


## 2. Return distribution

In [ ]:
ret = close.pct_change().dropna(how="all")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ret.stack().hist(bins=120, ax=ax[0], color="#2E86AB", alpha=0.85)
ax[0].set_title("Pooled daily-return distribution"); ax[0].set_xlabel("daily return")
ax[0].axvline(0, color="black", lw=0.7)

(ret.std() * np.sqrt(252)).sort_values().plot.barh(ax=ax[1], color="#E63946")
ax[1].set_title("Annualised vol per name"); ax[1].set_xlabel("vol"); plt.tight_layout(); plt.show()

print("pooled stats:")
s = ret.stack()
print(f"  mean = {s.mean():+.5f}  std = {s.std():.4f}  skew = {s.skew():+.2f}  kurt = {s.kurt():+.2f}")
print(f"  worst day = {s.min():+.2%}  best day = {s.max():+.2%}")


## 3. Cross-sectional dispersion over time
Dispersion = std of one-day returns across names. When dispersion is high, *cross-sectional* alpha has more room; when it collapses, factors have nothing to rank against.

In [ ]:
disp = ret.std(axis=1) * np.sqrt(252)
fig, ax = plt.subplots(figsize=(11, 3.5))
disp.plot(ax=ax, color="#bbbbbb", lw=0.7, label="daily")
disp.rolling(63).mean().plot(ax=ax, color="#2E86AB", lw=2.0, label="63d mean")
ax.set_ylabel("annualised x-sec std"); ax.set_title("Cross-sectional return dispersion"); ax.legend(); plt.show()


## 4. Universe correlation structure

In [ ]:
corr = ret.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"shrink": 0.7}, ax=ax, xticklabels=True, yticklabels=True)
ax.set_title(f"Pairwise daily-return correlation -- mean off-diag = {(corr.values[~np.eye(len(corr), dtype=bool)]).mean():.3f}")
plt.show()


## 5. Benchmark vs equal-weight universe

In [ ]:
bench = panels["benchmark_close"].iloc[:, 0].pct_change().dropna()
ew = ret.mean(axis=1)
df = pd.concat([bench.rename("SPY"), ew.rename("equal-weight universe")], axis=1).dropna()
((1 + df).cumprod()).plot(figsize=(11, 4))
plt.yscale("log"); plt.title("Equity curves -- benchmark vs equal-weight"); plt.ylabel("growth of $1"); plt.show()

print("annualised return / vol / Sharpe:")
for c in df:
    s = df[c]
    ann = (1 + s).prod() ** (252 / len(s)) - 1
    vol = s.std() * np.sqrt(252)
    print(f"  {c:24s}  ann={ann:+.2%}  vol={vol:.2%}  sharpe={ann/vol:+.2f}")


### Key takeaways

* Universe is well-aligned, no major missingness post-warmup.
* Pooled daily returns are fat-tailed (positive excess kurtosis) and slightly right-skewed.
* Cross-sectional dispersion oscillates with macro stress -- this is the playing field for our factors.
* Pairwise correlations are mostly +0.3-0.6 (typical for US large-cap), so a single tilt won't diversify much.
* SPY and the equal-weight universe trade together; we will benchmark vs SPY.